# Six Sigma DMAIC — Phase 5: Control (控制阶段)

> **数据源**: `SigmaFlow/templates/DMAIC_05_Control_Template.xlsx`
> **核心工具**:
> 1. SPC 统计过程控制 (单值-移动极差控制图 I-MR Chart 与失控判异)
> 2. 过程控制计划表图片 (Control Plan Table)
> 3. 防错机制登记清单图片 (Poka-Yoke Register Table)
> 4. 项目结案审计与签署清单图片 (Project Sign-off Table)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd(), Path.cwd() / "SigmaFlow", Path.cwd().parent] if (p / "minitab_dmaic_visuals.py").exists()), Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

import importlib
import minitab_dmaic_visuals as mv
importlib.reload(mv)
mv.apply_minitab_theme()

TEMPLATE_PATH = PROJECT_ROOT / "templates" / "DMAIC_05_Control_Template.xlsx"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loading Control Template from: {TEMPLATE_PATH}")


## 5.1 在线统计过程控制 (I-MR Control Chart)
读取 `SPC_Monitoring_Data` 工作表，绘制改进后的生产监控数据，验证稳定性并检测异常点。


In [4]:
df_spc = pd.read_excel(TEMPLATE_PATH, sheet_name="SPC_Monitoring_Data")
spc_series = df_spc["Dimension_mm"].values
out_spc_img = FIG_DIR / "phase5_imr_control_chart.png"

fig_spc = mv.plot_spc_xmr_chart(
    spc_series,
    title_x="I-Chart: Individual Measurements (dimension_mm)",
    title_mr="MR-Chart: Moving Range (Short-term Variation)",
    output_path=str(out_spc_img)
)
display(Image(filename=str(out_spc_img)))


## 5.2 计量型子组控制图 (Xbar-R Control Chart)
针对批量连续加工场景，采用子组抽样（Subgroups $n=5$），同时监控**过程中心（均值 $\bar{X}$）** 与 **过程波动（极差 $R$）**：
- **$R$ 控制图（波动监控）**: 验证短期组内离散是否稳定（$UCL = D_4 \bar{R}, LCL = D_3 \bar{R}$）
- **$\bar{X}$ 控制图（均值监控）**: 监控加工均值是否发生中心漂移（$UCL = \bar{\bar{X}} + A_2 \bar{R}, LCL = \bar{\bar{X}} - A_2 \bar{R}$）
- **失控判定**: 自动检测并标红超出 3σ 限的异常点（标注 Minitab Test 1 标识）。


In [6]:
df_xbar = pd.read_excel(TEMPLATE_PATH, sheet_name="SPC_Subgroup_XbarR")
subgroup_cols = [c for c in df_xbar.columns if c.startswith("Sample_")]
subgroup_data = df_xbar[subgroup_cols].values

out_xbar_r_img = FIG_DIR / "phase5_xbar_r_control_chart.png"
mv.plot_spc_xbar_r_chart(
    subgroup_data,
    title_xbar="Xbar Chart of Dimension_mm (Subgroup Size n=5)",
    title_r="R Chart of Dimension_mm (Subgroup Ranges)",
    xlabel="Subgroup",
    output_path=str(out_xbar_r_img)
)
display(Image(filename=str(out_xbar_r_img)))


## 5.3 计数型控制图全谱系 (Attributes Control Charts: P, NP, C, U Charts)
针对不合格品率（Defectives）与缺陷数（Defects），分别应用 SPC 计数型控制图体系：
1. **P 控制图 (不合格品率图)**: 适用于样本量可变或恒定的不合格品比例 $p = d/n$ 监控。
2. **NP 控制图 (不合格品数图)**: 适用于样本量恒定 $n$ 的不合格品绝对件数 $np$ 监控。
3. **C 控制图 (缺陷数图)**: 适用于检验单元恒定（如固定单件/单板）的总缺陷数 $c$ 监控。
4. **U 控制图 (单位缺陷数图)**: 适用于检验面积/单元数可变的单位缺陷率 $u = c/n$ 监控。


In [8]:
df_p_np = pd.read_excel(TEMPLATE_PATH, sheet_name="SPC_Attribute_P_NP")
df_c_u = pd.read_excel(TEMPLATE_PATH, sheet_name="SPC_Attribute_C_U")

out_p_img = FIG_DIR / "phase5_p_chart.png"
out_np_img = FIG_DIR / "phase5_np_chart.png"
out_c_img = FIG_DIR / "phase5_c_chart.png"
out_u_img = FIG_DIR / "phase5_u_chart.png"

# 1. P 控制图
mv.plot_spc_p_chart(
    defectives=df_p_np["Defective_Count_d"].values,
    sample_sizes=df_p_np["Sample_Size_n"].values,
    title="P Chart of Defectives (不合格品率控制图)",
    xlabel="Inspection Batch",
    ylabel="Proportion Defective",
    output_path=str(out_p_img)
)

# 2. NP 控制图
mv.plot_spc_np_chart(
    defectives=df_p_np["Defective_Count_d"].values[:25],
    sample_size=100,
    title="NP Chart of Defectives (不合格品数控制图, n=100)",
    xlabel="Inspection Batch",
    ylabel="Defective Count (np)",
    output_path=str(out_np_img)
)

# 3. C 控制图
mv.plot_spc_c_chart(
    defect_counts=df_c_u["Total_Defects_c"].values[:25],
    title="C Chart of Defects (缺陷数控制图)",
    xlabel="Inspection Run",
    ylabel="Total Defect Count (c)",
    output_path=str(out_c_img)
)

# 4. U 控制图
mv.plot_spc_u_chart(
    defect_counts=df_c_u["Total_Defects_c"].values,
    units_inspected=df_c_u["Units_Inspected_n"].values,
    title="U Chart of Defects per Unit (单位缺陷数控制图)",
    xlabel="Inspection Run",
    ylabel="Defects per Unit (u)",
    output_path=str(out_u_img)
)

print("--- [1] 不合格品率与不合格品数控制图 (P & NP) ---")
display(Image(filename=str(out_p_img)))
display(Image(filename=str(out_np_img)))

print("--- [2] 缺陷数与单位缺陷数控制图 (C & U) ---")
display(Image(filename=str(out_c_img)))
display(Image(filename=str(out_u_img)))


## 5.4 统计过程控制计划表图片 (Process Control Plan Card)
读取 `Control_Plan` 工作表，输出闭环控制计划数据表卡片图片。


In [10]:
df_cplan = pd.read_excel(TEMPLATE_PATH, sheet_name="Control_Plan")
out_cplan_img = FIG_DIR / "phase5_control_plan_table.png"

mv.render_table_card(
    df_cplan,
    title="Process Control Plan & Out-of-Control Action Plan (OCAP)",
    footer_lines=["Control Plan fully integrated into Manufacturing Execution System (MES)."],
    output_path=str(out_cplan_img),
    figsize=(11.0, 5.5)
)
display(Image(filename=str(out_cplan_img)))


## 5.5 防错装置与技术措施 (Poka-Yoke Register Table Card)
读取 `Poka_Yoke_Register` 工作表，输出硬件电气防错台账数据表图片。


In [12]:
df_poka = pd.read_excel(TEMPLATE_PATH, sheet_name="Poka_Yoke_Register")
out_poka_img = FIG_DIR / "phase5_poka_yoke_table.png"

mv.render_table_card(
    df_poka,
    title="Poka-Yoke Mistake-Proofing Device Register Table",
    footer_lines=["Poka-Yoke interlocks verified during shift startup check."],
    output_path=str(out_poka_img),
    figsize=(10.5, 4.5)
)
display(Image(filename=str(out_poka_img)))


## 5.6 六西格玛项目审计结案签署 (Project Sign-Off Table Card)
读取 `Project_Signoff` 工作表，输出带核准结论的最终财务效益与签署表图片。


In [14]:
df_signoff = pd.read_excel(TEMPLATE_PATH, sheet_name="Project_Signoff")
out_signoff_img = FIG_DIR / "phase5_project_signoff_table.png"

mv.render_table_card(
    df_signoff,
    title="Six Sigma Project Final Audit & Sign-Off Checklist",
    footer_lines=["Approved by Champion, Financial Controller, and Process Owner."],
    output_path=str(out_signoff_img)
)
display(Image(filename=str(out_signoff_img)))


### 📌 Control 阶段总结
1. **闭环维稳**: 部署 I-MR 控制图在线监控，配合 OCAP 应急反应机制。
2. **防错固化**: 落实主轴油温硬件联锁与气压报警，成果长效保持。
3. **所有过程控制计划、防错和结案表均生成标准 Minitab 图片归档**。
